# ads-bib Colab quickstart

This notebook runs the full `ads-bib` pipeline in Google Colab with a T4 GPU. It uses public Hugging Face models, so the only required key is a NASA ADS token.

Before running: choose `Runtime > Change runtime type > T4 GPU`.


## 1. Install

This installs the package and the matching CUDA Torch/TorchVision runtime used by the notebook.


In [1]:
%pip install -q uv
!uv pip install --system -q --extra-index-url https://download.pytorch.org/whl/cu124 \
    ads-bib "torch==2.6.0" "torchvision==0.21.0" bitsandbytes
!uv pip uninstall --system -q torchcodec || true

## 2. Check the GPU

If this cell fails, switch the Colab runtime to T4 GPU and run the notebook again.


In [2]:
import logging
import warnings

import torch
import transformers

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU found. In Colab choose Runtime > Change runtime type > T4 GPU, "
        "then run this notebook again."
    )

print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

warnings.filterwarnings("ignore")
logging.getLogger("torchao").setLevel(logging.ERROR)
transformers.logging.set_verbosity_error()


## 3. Add tokens

Create an ADS token at [ADS token settings](https://ui.adsabs.harvard.edu/user/settings/token). A Hugging Face token is optional and can make downloads more reliable in Colab.

`/content/ads-bib-colab` is temporary. Download anything you want to keep before the Colab session ends.


In [3]:
from getpass import getpass
import os
from pathlib import Path

PROJECT_ROOT = "/content/ads-bib-colab"
Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

ads_token = getpass("ADS_TOKEN (https://ui.adsabs.harvard.edu/user/settings/token): ").strip()
if not ads_token:
    raise ValueError("ADS_TOKEN is required.")
os.environ["ADS_TOKEN"] = ads_token

hf_token = None
try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = getpass("HF_TOKEN optional, Enter to skip: ").strip() or None

if hf_token:
    from huggingface_hub import login

    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
    print("HF token loaded.")
else:
    print("No HF token; continuing with public downloads.")

print(f"Project folder: {PROJECT_ROOT}")


## 4. Choose the ADS search

The notebook starts from the `local_gpu` preset and only changes the three model roles for this public Colab run.


In [4]:
from ads_bib.presets import preset_to_dict

SEARCH_QUERY = 'author:"Hawking, S*"'
RUN_NAME = "ads_bib_colab_hawking"

CONFIG = preset_to_dict("local_gpu")
CONFIG["run"]["run_name"] = RUN_NAME
CONFIG["run"]["project_root"] = PROJECT_ROOT
CONFIG["search"]["query"] = SEARCH_QUERY
CONFIG["translate"].update(
    {
        "provider": "nllb",
        "model": "JustFrederik/nllb-200-distilled-600M-ct2-int8",
    }
)
CONFIG["topic_model"].update(
    {
        "embedding_model": "Qwen/Qwen3-Embedding-0.6B",
        "llm_model": "Qwen/Qwen3-4B-Instruct-2507",
    }
)

print("Search query:", SEARCH_QUERY)
print("Translation:", CONFIG["translate"]["model"])
print("Embeddings:", CONFIG["topic_model"]["embedding_model"])
print("Topic labels:", CONFIG["topic_model"]["llm_model"])


## 5. Run the pipeline

This runs search, export, translation, tokenization, embeddings, topic modeling, visualization, curation, and citations.


In [10]:
from ads_bib.runner import load_run_config, run_resolved_config

resolved_config = load_run_config(
    config=CONFIG,
    project_root="/content/ads-bib-colab",
)

result = run_resolved_config(
    resolved_config,
    project_root="/content/ads-bib-colab",
    run_name="ads_bib_colab_hawking",
    output_mode="notebook",
)

print("Run finished.")
print(f"Run folder: {result.run.paths['root']}")

## 6. Results

The most useful outputs are the run folder, the topic map, the ADS export files, and the citation files.


In [6]:
cols = [c for c in ["Bibcode", "Year", "Author", "Title"] if c in result.publications.columns]

result.publications[cols].head(10)
result.topic_info.head(20)

In [ ]:
run_root = result.run.paths["root"]
topic_map = result.run.paths["plots"] / "topic_map.html"
export_dir = result.run.paths["export"]
citations_dir = result.run.paths["citations"]

print(f"Run folder: {run_root}")
print(f"Export files: {export_dir}")
print(f"Citation files: {citations_dir}")
print(f"Topic map: {topic_map}")

topic_map_obj = getattr(result, "topic_map", None)
if topic_map_obj is None and result.topic_df is not None:
    from ads_bib.visualize import create_topic_map

    vis_config = CONFIG.get("visualization", {})
    topic_map_obj = create_topic_map(
        result.topic_df,
        title=vis_config.get("title", "ADS Topic Map"),
        subtitle=vis_config.get("subtitle", ""),
        dark_mode=vis_config.get("dark_mode", True),
        font_family=vis_config.get("font_family", "Cinzel"),
        topic_tree=vis_config.get("topic_tree", False),
    )

topic_map_obj